# ESP32-S3 Camera Integration Plan

This notebook documents a robotics-first integration plan for the ESP32-S3 camera node, the base controller, and the arm controller.

Goals for this design:
- Use ESP-NOW for low-latency command and feedback exchange.
- Keep camera pose estimation and QR decoding on the camera.
- Keep live video streaming separate from control and pose computation.
- Give the base controller ownership of autonomous pickup sequencing.

## What The Camera Firmware Already Does

The standalone camera project in `firmware/cam_stream/src/main.cpp` is already close to the right architecture for robotics use. It currently:

- Captures frames from the ESP32-S3 camera.
- Converts frames to grayscale and runs quirc QR detection.
- Computes QR pose from a homography-based model.
- Tracks detections over time with confidence decay and short-term hold.
- Publishes a JSON API at `/data` and health info at `/status`.
- Exposes `/stream` as an MJPEG view for laptop visualization.
- Sends a pose packet to an arm endpoint through a WiFi UDP helper today.

The PDF study file confirms the intended math path: camera projection, planar homography, homography decomposition, Euler extraction, confidence gating, and a downstream IK packet.

## Recommended Control Architecture

For low latency and predictable behavior, the clean split is:

1. Base controller owns autonomy state.
2. Base sends a request to the camera when the user presses the autonomous pickup button.
3. Camera runs QR detection and pose estimation locally.
4. Camera replies with the latest pose, QR text, confidence, and estimated flag.
5. Base decides whether the robot should reposition, re-query, or hand off to the arm.
6. Base sends the arm a final execute command only after pose is good enough for IK.

This keeps the base as the single place where navigation, reachability checks, and safety decisions live. The camera should not decide motion policy. It should only produce vision evidence.

## Transport Recommendation

Use ESP-NOW for the control plane between base and camera, and between base and arm. That gives you the lowest practical latency on the ESP32s without depending on a router or cloud service.

Keep HTTP only for the camera's display path:

- `/stream` for a lightweight live preview.
- `/status` for diagnostics.
- `/data` only for debugging or a local dashboard.

Do not make the autonomous pickup flow depend on the web server. The control path should still work even if the laptop dashboard is closed.

## Platform Pose For Box Placement

The camera can also support a box-placement target: a squared tall platform that the arm should place the box onto.

Best practical options, in order:

1. Put a square fiducial on the platform top or front face, then estimate pose from its corners.
2. If no fiducial is allowed, detect the square top face and use the known platform size to recover pose.
3. If the platform is visually plain and repetitive, add a high-contrast marker because pure square-edge detection will be less reliable.

For the control flow, the base should ask the camera for a second pose once it is close enough. The reply should include:

- platform_id or platform_label
- pose_valid
- translation and orientation in camera frame
- confidence
- estimated flag

Then the base can compute a place pose with a fixed vertical offset above the platform top, verify reachability, and only then send the arm the final place command.

In [3]:
def decide_platform_action(confidence, pose_valid, estimated, height_error_mm):
    if not pose_valid or confidence < 0.25:
        return 'SEARCH_FOR_PLATFORM'
    if estimated and confidence < 0.60:
        return 'REACQUIRE_PLATFORM_POSE'
    if abs(height_error_mm) > 40:
        return 'ADJUST_BASE_OR_APPROACH'
    if confidence < 0.85:
        return 'VERIFY_PLACE_POSE'
    return 'SEND_ARM_PLACE_COMMAND'

In [1]:
from dataclasses import dataclass


@dataclass
class CameraCommand:
    task_id: int
    mode: str
    target_color: str = 'UNKNOWN'
    request_pose: bool = True
    stream_only: bool = False


@dataclass
class CameraReply:
    task_id: int
    frame_id: int
    qr_text: str
    confidence: float
    decoded: bool
    estimated: bool
    tx_mm: float
    ty_mm: float
    tz_mm: float
    roll_deg: float
    pitch_deg: float
    yaw_deg: float


camera_cmd = CameraCommand(task_id=1, mode='autonomous_pick', target_color='RED')
camera_reply = CameraReply(
    task_id=1,
    frame_id=42,
    qr_text='RED',
    confidence=0.91,
    decoded=True,
    estimated=False,
    tx_mm=120.0,
    ty_mm=-35.0,
    tz_mm=430.0,
    roll_deg=2.0,
    pitch_deg=-1.5,
    yaw_deg=4.2,
)
camera_reply

CameraReply(task_id=1, frame_id=42, qr_text='RED', confidence=0.91, decoded=True, estimated=False, tx_mm=120.0, ty_mm=-35.0, tz_mm=430.0, roll_deg=2.0, pitch_deg=-1.5, yaw_deg=4.2)

## Autonomous Pickup State Machine

A small state machine is the right control structure for the base. Suggested states:

- `IDLE`
- `REQUEST_SCAN`
- `WAIT_CAMERA_REPLY`
- `ALIGN_BASE`
- `VERIFY_FINAL_POSE`
- `SEND_ARM_EXECUTE`
- `WAIT_ARM_DONE`
- `COMPLETE`
- `FAIL_SAFE`

Transition rule example:

- If confidence is high and pose is valid, advance to arm execution.
- If confidence is medium, reposition the base and request another camera update.
- If confidence is low or stale, stop motion and re-enter scan.

This is better than a blocking while-loop because it stays responsive to fresh camera data and user cancellation.

In [2]:
def decide_next_action(confidence, pose_valid, estimated, age_ms):
    if not pose_valid or confidence < 0.20:
        return 'SAFE_STOP_AND_RESCAN'
    if estimated and age_ms > 1000:
        return 'HOLD_POSITION_AND_REQUERY'
    if confidence < 0.55:
        return 'REPOSITION_BASE'
    if confidence < 0.85:
        return 'VERIFY_FINAL_POSE'
    return 'SEND_ARM_EXECUTE'


decide_next_action(0.91, True, False, 120)

'SEND_ARM_EXECUTE'

## Streaming Strategy

The camera should expose a low-computation live view mode that is display-only. The stream should not be the place where pose math or QR decode happens.

Recommended behavior:

- Lower frame rate for preview than for control.
- Smaller frame size if the dashboard only needs situational awareness.
- Keep the vision pipeline independent from the HTTP stream handler.
- Use the stream only on the local website for human monitoring.

If bandwidth or CPU becomes tight, reduce preview quality before touching the pose pipeline. The robot should lose display quality before it loses control quality.

## Robotics Engineer Recommendations

My recommendation is to keep the vision node simple and deterministic:

- Camera handles sensing and local pose estimation.
- Base handles planning, thresholding, and retry logic.
- Arm handles IK execution and the hardcoded drop actions.
- ESP-NOW carries the time-critical packets.
- HTTP carries only preview and diagnostics.

The main engineering risk is mixing a slow display path with a time-critical control path. Avoid that by separating packet types and by never making the autonomous sequence wait on the stream server.

Next practical implementation step: define the ESP-NOW packet structs for base->camera, camera->base, and base->arm, then replace the camera's current UDP send with an ESP-NOW response packet.